# Phase 3: Equipment Failure Classification (설비 장애 분류 및 하이퍼파라미터 최적화)

이 단계에서는 전처리 및 오버샘플링을 마친 CMP 가공 데이터를 활용하여 실제 설비 고장을 감지하는 분류 머신러닝 알고리즘을 훈련합니다. 강력한 트리 앙상블 기법인 **XGBoost**와 **LightGBM**을 비교 분석하고, 최첨단 탐색 엔진인 **Optuna**를 사용하여 모델 성능을 자동으로 튜닝합니다.

## 🎯 실습 목표
1. **학습 데이터 바인딩**: Phase 2의 SMOTE 완료 훈련셋 및 테스트셋 로드
2. **앙상블 분류 모델 훈련**: XGBoost 및 LightGBM 분류기 기본 파라미터 학습
3. **다차원 모델 검증**: Confusion Matrix 플로팅 및 F1-Score 정밀 비교
4. **Optuna 하이퍼파라미터 튜닝**: 성능 최적화를 위한 최적 파라미터 자동 탐색 루프

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score
import xgboost as xgb
import lightgbm as lgb
import optuna

# 경고 무시 및 테마 설정
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 데이터 로드
try:
    X_train = pd.read_csv('./data/cmp_X_train_res.csv')
    X_test = pd.read_csv('./data/cmp_X_test.csv')
    y_train = pd.read_csv('./data/cmp_y_train_res.csv')['failure']
    y_test = pd.read_csv('./data/cmp_y_test.csv')['failure']
    print("훈련 및 검증 데이터셋 로드 완료!")
except FileNotFoundError:
    print("데이터 파일이 없습니다. 02_imbalance_and_features.ipynb를 완료해 주세요.")

## 1. 머신러닝 분류기 학습 (XGBoost vs LightGBM)
두 가지 대표적인 고성능 그라디언트 부스팅 라이브러리로 모델을 기본 학습시킵니다.

In [ ]:
# 1. XGBoost 학습
xgb_model = xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

# 2. LightGBM 학습
lgb_model = lgb.LGBMClassifier(random_state=42, verbose=-1)
lgb_model.fit(X_train, y_train)

print("XGBoost 및 LightGBM 기본 학습 완료")

## 2. 혼동 행렬 (Confusion Matrix) 및 정밀 성능 비교
테스트셋을 대상으로 정밀도(Precision)와 재현율(Recall) 지표를 검증하고 시각화합니다.

In [ ]:
xgb_preds = xgb_model.predict(X_test)
lgb_preds = lgb_model.predict(X_test)

print("=== XGBoost 분류 성능 보고서 ===")
print(classification_report(y_test, xgb_preds))

# 혼동 행렬 시각화
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(confusion_matrix(y_test, xgb_preds), annot=True, fmt='d', cmap='Blues', ax=ax[0])
ax[0].set_title('XGBoost Confusion Matrix')
ax[0].set_xlabel('Predicted')
ax[0].set_ylabel('Actual')

sns.heatmap(confusion_matrix(y_test, lgb_preds), annot=True, fmt='d', cmap='Greens', ax=ax[1])
ax[1].set_title('LightGBM Confusion Matrix')
ax[1].set_xlabel('Predicted')
ax[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

## 3. Optuna 기반 하이퍼파라미터 최적화
부스팅 모델의 핵심 파라미터들을 효과적으로 조합하여 F1-score 지표를 최대로 정렬하기 위한 튜닝 파이프라인을 구축합니다.

In [ ]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'eval_metric': 'logloss',
        'random_state': 42
    }
    
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train)
    
    # 검증 데이터셋에 대한 F1-score 최적화 목표 설정
    preds = model.predict(X_test)
    score = f1_score(y_test, preds)
    
    return score

# 튜닝 태스크 개시
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15, timeout=120) # 15회 빠른 반복 실습

print("\n--- 최적화 완료 ---")
print(f"최적 F1-Score: {study.best_value:.4f}")
print("최적 하이퍼파라미터:", study.best_params)

# 최적 모델 저장 (XAI 적용용)
best_xgb = xgb.XGBClassifier(**study.best_params, random_state=42, eval_metric='logloss')
best_xgb.fit(X_train, y_train)

import joblib
joblib.dump(best_xgb, './data/cmp_best_xgb_model.pkl')
print("\n최적 XGBoost 모델 파일 ('./data/cmp_best_xgb_model.pkl') 저장 완료")